In [3]:
# -*- coding: utf-8 -*-
"""
moonshot_compounding — Stage 3 Production Pipeline
PIT Universe + Raw Price Panel Construction (yfinance + Bloomberg)

This is a CURATED, production-ordered version of the full session
transcript, kept in sync with build_log.md.

REMOVED from the original session transcript, deliberately, and why:
  - The custom split/bonus adjustment logic ("Step 3/4" in the original
    session) and its validation. This logic was PROVEN WRONG — it
    double-adjusts prices that yfinance's Close column already adjusts
    for splits. See build_log.md section 2 for the full evidence
    (7 real split events, 5 tickers, confirmed near-0% actual price
    change at each split boundary vs the large mechanical drop a truly
    raw series would show). DO NOT reintroduce this logic. The output
    file it produced (adjusted/prices_adjusted_pilot_v1.parquet) was
    deleted and should stay deleted.
  - Three diagnostic-only scripts ("Direct Test", "Direct Test
    Generalized", "Widened Pilot") that were used to GATHER the evidence
    above. They don't produce pipeline data and don't need to be re-run;
    their conclusions are captured in build_log.md.
  - A duplicate copy of the PIT cross-validation script (Stage 3-QA2)
    that appeared a second time, verbatim, later in the session. Kept
    only ONE copy here — re-run this same cell again whenever you need
    to re-validate, rather than maintaining two copies that could drift.

FIXES APPLIED in this version (beyond the removals above):
  - Stage 3C's skip-set logic previously counted ANY ticker that EVER
    had a 'success' row, not each ticker's LATEST status. This meant a
    ticker that succeeded once (e.g. a phantom 1-row quote) and was
    later correctly reclassified to 'failed' would be silently,
    permanently skipped by every future run — never retried. Fixed to
    deduplicate to latest status first, consistent with the rest of
    the pipeline.
  - Added a generalized SPARSE-TICKER RECLASSIFICATION step (after the
    Bloomberg merge, before Stage 3D) that checks every ticker's
    ABSOLUTE valid-row count (not just NaN %) against the longest
    planned feature window. Catches cases like DENABANK.NS (226 rows,
    97% NaN, only 6 usable) automatically on any future re-run, not
    just this one hardcoded name.

Run the cells below in order, top to bottom, in Colab.
"""

# ----------------------------------------------------------------------
# STEP 0: Mount Drive and set up the folder structure we agreed on.
# We create every folder now, even ones we won't populate this session,
# so future sessions can just write into an already-existing structure
# instead of each session reinventing where things go.
# ----------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'

folders = [
    'raw/universe',
    'raw/prices',
    'corporate_actions',
    'adjusted',
    'universe',
    'features',
    'validation',
]
for f in folders:
    os.makedirs(BASE + f, exist_ok=True)

print("Folder structure created under:", BASE)
for f in folders:
    print(" -", BASE + f)

# ======================================================================
# STAGE 3A — Load or build the canonical PIT universe file.
#
# This file was originally built and inspected in a separate sandbox
# during our earlier diagnostic sessions. It has NOT yet been built
# inside THIS Colab environment. This cell checks if it already exists
# on Drive (e.g. if you've run this before) and builds it fresh from
# the Bloomberg Sheet4 Excel file if not.
# ======================================================================
import pandas as pd
import numpy as np
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
UNIVERSE_PATH = BASE + 'universe/canonical_membership_semiannual.parquet'

if os.path.exists(UNIVERSE_PATH):
    print(f"Found existing canonical universe file at {UNIVERSE_PATH} — loading it.")
    canonical = pd.read_parquet(UNIVERSE_PATH)
else:
    print("Canonical universe file not found on Drive — building it now from Sheet4.")
    print("You will be prompted to upload NSE_historical_constituents.xlsx.")
    from google.colab import files
    uploaded = files.upload()
    xlsx_path = list(uploaded.keys())[0]

    import openpyxl
    import datetime as dt

    wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb['Sheet4']
    rows = list(ws.iter_rows(values_only=True))
    header_row = rows[1]
    # Sheet4 is a wide, block-structured layout: every 6th column starts a
    # new snapshot date block (date, id(), name(), id_isin(), id_exch_symbol()).
    date_positions = [(i, v) for i, v in enumerate(header_row) if isinstance(v, dt.datetime)]

    records = []
    for offset, snap_date in date_positions:
        id_col, name_col, isin_col, exch_col = offset + 1, offset + 2, offset + 3, offset + 4
        for r in rows[2:]:
            if id_col >= len(r):
                continue
            bbg_id = r[id_col]
            if bbg_id is None:
                continue
            records.append((
                snap_date,
                bbg_id,
                r[name_col] if name_col < len(r) else None,
                r[isin_col] if isin_col < len(r) else None,
                r[exch_col] if exch_col < len(r) else None,
            ))

    df = pd.DataFrame(records, columns=['snap_date', 'bbg_id', 'name', 'isin', 'exch_symbol'])
    df['snap_date'] = pd.to_datetime(df['snap_date'])

    # KNOWN ISSUE: the 2018-06-30 snapshot in the raw file is truncated
    # (353 rows instead of ~500 — a partial-pull artifact, confirmed by
    # checking it starts alphabetically at "ABAN", missing names like
    # "3M", "360ONE", "63MOONS" present in every neighboring snapshot).
    # We substitute the 2017-12-31 snapshot as a documented stand-in.
    # Quantified risk (see build_log.md): comparing the two real snapshots
    # bracketing this gap (2017-12-31 vs 2018-12-31) shows 56 additions +
    # 56 removals combined across BOTH 2018 rebalances — an upper bound of
    # ~22% of the universe potentially misclassified specifically in the
    # 2018-07-01 to 2018-12-30 window. This is a real, non-trivial risk,
    # not a rounding footnote — flagged explicitly here and in build_log.md
    # so any future test result concentrated in that window gets scrutiny.
    snap_2017_12 = df[df['snap_date'] == pd.Timestamp('2017-12-31')].copy()
    carried = snap_2017_12.copy()
    carried['snap_date'] = pd.Timestamp('2018-06-30')
    carried['snapshot_source'] = 'carried_forward_from_2017-12-31'

    df_clean = df[df['snap_date'] != pd.Timestamp('2018-06-30')].copy()
    df_clean['snapshot_source'] = 'observed'

    canonical = pd.concat([df_clean, carried], ignore_index=True)
    canonical = canonical.sort_values(['snap_date', 'isin']).reset_index(drop=True)

    canonical.to_parquet(UNIVERSE_PATH, index=False)
    print(f"Built and saved canonical universe to {UNIVERSE_PATH}")

print(f"\nCanonical universe loaded: {canonical.shape[0]} rows, "
      f"{canonical['isin'].nunique()} unique ISINs, "
      f"{canonical['snap_date'].nunique()} snapshot dates.")


# ======================================================================
# STAGE 3B — Derive the pull ticker list from the canonical universe.
#
# We confirmed earlier that every ISIN in this dataset maps to exactly
# ONE exch_symbol across all 33 snapshots — no ticker-rename cases exist.
# That means the crosswalk is simple: one ISIN -> one ticker.
#
# TWO THINGS DISCOVERED WHILE TESTING THIS LOGIC (both fixed below):
#
# 1. Missing symbols are stored as the LITERAL STRING "#N/A" in this
#    file, not as a true null. pandas' isna()/dropna() do NOT catch a
#    string that merely SAYS "#N/A" — we have to check for both forms
#    explicitly, or 5 bogus rows silently collapse into one fake
#    "#N/A.NS" ticker and corrupt the pull list with duplicates.
#
# 2. A stock can have a valid symbol on MOST snapshots but "#N/A" on
#    just one or two (e.g. 360 ONE WAM Ltd — formerly IIFL Wealth
#    Management — shows "#N/A" only during its 2023 rebrand window,
#    with a valid "360ONE" symbol on 11 of its 13 appearances). Such a
#    stock is NOT actually unpullable — we must only exclude an ISIN if
#    it has NO valid symbol on ANY snapshot, not if it merely has a gap
#    on some snapshots.
# ======================================================================

is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')

# Pullable: the most recent VALID (non-missing) symbol per ISIN,
# ignoring any "#N/A"/null rows entirely when picking the latest one.
isin_symbol_map = (
    canonical[~is_missing_row]
    .sort_values('snap_date')
    .groupby('isin')
    .last()[['exch_symbol', 'name']]
    .reset_index()
)
isin_symbol_map['yf_ticker'] = isin_symbol_map['exch_symbol'].str.strip() + '.NS'

# Truly excluded: ISINs with NO valid symbol on ANY snapshot at all —
# i.e. ISINs that never made it into isin_symbol_map above. This is
# NOT the same as "ISINs with a missing symbol on at least one date."
all_isins = set(canonical['isin'].unique())
pullable_isins = set(isin_symbol_map['isin'])
truly_excluded_isins = all_isins - pullable_isins

excluded = canonical[canonical['isin'].isin(truly_excluded_isins)][['isin', 'name']].drop_duplicates()
excluded['reason'] = 'no_valid_exch_symbol_on_any_snapshot_likely_unlisted_or_newly_demerged_entity'
os.makedirs(BASE + 'validation', exist_ok=True)
excluded.to_csv(BASE + 'validation/excluded_tickers_no_symbol.csv', index=False)

print(f"\nDerived {len(isin_symbol_map)} pullable tickers from Sheet4.")
print(f"Excluded {len(excluded)} ISINs with NO valid symbol on any snapshot:")
print(excluded.to_string(index=False))

dupes = isin_symbol_map[isin_symbol_map['yf_ticker'].duplicated(keep=False)]
assert dupes.empty, f"Found duplicate yf_tickers — investigate before proceeding:\n{dupes}"
assert len(isin_symbol_map) + len(excluded) == canonical['isin'].nunique(), \
    "Pullable + excluded counts don't reconcile with total unique ISINs — stop and investigate"
print("\n[PASS] Pullable + excluded counts reconcile exactly with total unique ISINs. No duplicate tickers.")

pull_list = isin_symbol_map[['isin', 'yf_ticker', 'name']].copy()
pull_list.to_csv(BASE + 'universe/pull_ticker_list_v1.csv', index=False)
print(f"\nSaved pull list ({len(pull_list)} tickers) to universe/pull_ticker_list_v1.csv")
print("\nSample of pull list:")
print(pull_list.head(10).to_string(index=False))

# ======================================================================
# STAGE 3C — Bulk pull: raw OHLCV + corporate actions for the full
# universe (1,071 tickers), 2010-01-01 to present.
#
# DESIGN, explained:
#
# - One ticker = one atomic unit of work. Each ticker's price data is
#   saved to its OWN small parquet file (raw/prices/batches/<ticker>.parquet),
#   and its OWN split data likewise (corporate_actions/batches/<ticker>.parquet).
#   This means a crash or Colab disconnect never corrupts a big shared
#   file — at worst we lose whatever ticker was mid-download when it happened.
#
# - A checkpoint CSV (_pull_progress.csv) records every ticker's outcome:
#   success/failed, row count, error message, timestamp. Re-running this
#   cell SKIPS any ticker already marked 'success' — so this script is
#   safely re-runnable as many times as needed until everything succeeds,
#   without ever re-downloading data we already have.
#
# - Each ticker gets up to 3 attempts (1 initial + 2 retries) with a pause
#   between retries, before being logged as a genuine failure. Genuine
#   failures are NOT silently dropped — they stay visible in the
#   checkpoint log with their error message, for us to investigate.
#
# - A deliberate 0.75s delay between tickers reduces the risk of Yahoo
#   rate-limiting us partway through a run of 1,000+ tickers.
#
# EXPECTED RUNTIME: with ~1,071 tickers and a ~1-2s round trip each
# (download + delay), this will likely take 30-60+ minutes. That's fine —
# because of the checkpointing, you can stop and resume this at any time
# by just re-running the cell.
# ======================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

BASE = '/content/drive/MyDrive/moonshot_compounding/'

START = '2010-01-01'
END = pd.Timestamp.today().strftime('%Y-%m-%d')  # pull through today

PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CA_BATCH_DIR = BASE + 'corporate_actions/batches/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'

os.makedirs(PRICE_BATCH_DIR, exist_ok=True)
os.makedirs(CA_BATCH_DIR, exist_ok=True)

DELAY_SECONDS = 0.75        # deliberate pause between EVERY ticker
MAX_ATTEMPTS = 3            # 1 initial attempt + 2 retries
RETRY_DELAY_SECONDS = 5     # longer pause specifically before a retry
CHECKPOINT_FLUSH_EVERY = 25 # save progress to disk every N tickers


# ------------------------------------------------------------
# Load or initialize the checkpoint log.
# ------------------------------------------------------------
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = pd.read_csv(CHECKPOINT_PATH)
    print(f"Resuming from existing checkpoint: {len(checkpoint)} tickers already attempted.")
else:
    checkpoint = pd.DataFrame(columns=['ticker', 'status', 'rows_pulled', 'n_splits', 'error_message', 'timestamp'])
    print("No existing checkpoint found — starting fresh.")

# FIX: the ORIGINAL version of this line was `set(checkpoint[checkpoint['status']
# == 'success']['ticker'])` — that counts ANY row ever marked 'success', not
# each ticker's LATEST status. A ticker that succeeded once (e.g. a phantom
# 1-row quote) and was later correctly reclassified to 'failed' would still
# satisfy that condition and get silently, permanently skipped by every
# future run of this cell — never retried again. Fixed to deduplicate to
# latest status per ticker first, consistent with every other part of this
# pipeline that reads the checkpoint.
latest_status_now = checkpoint.sort_values('timestamp').groupby('ticker').last()['status']
already_succeeded = set(latest_status_now[latest_status_now == 'success'].index)
print(f"{len(already_succeeded)} tickers already successfully pulled — these will be SKIPPED.\n")

pull_list_df = pd.read_csv(BASE + 'universe/pull_ticker_list_v1.csv')
tickers_to_pull = [t for t in pull_list_df['yf_ticker'].tolist() if t not in already_succeeded]

print(f"{len(tickers_to_pull)} tickers remaining to pull this run.\n")

new_checkpoint_rows = []


# ------------------------------------------------------------
# Main pull loop.
# ------------------------------------------------------------
for i, ticker in enumerate(tickers_to_pull):
    success = False
    last_error = None

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            df = yf.download(ticker, start=START, end=END, auto_adjust=False, progress=False)

            if df.empty:
                raise ValueError("yfinance returned an empty DataFrame (ticker may be delisted, "
                                  "merged, or an invalid/incorrect symbol)")

            # Same MultiIndex-flatten fix confirmed necessary in the pilot —
            # every single-ticker yf.download() call needs this.
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['ticker'] = ticker
            df = df.reset_index()

            safe_name = ticker.replace('.', '_')
            df.to_parquet(PRICE_BATCH_DIR + f'{safe_name}.parquet', index=False)

            # Corporate actions for the same ticker, same successful attempt.
            # Kept on file even though we don't need it for price adjustment
            # (confirmed this session: yfinance's Close is already split-
            # adjusted) — useful later if a total-return series is ever needed.
            t_obj = yf.Ticker(ticker)
            splits = t_obj.splits
            n_splits = 0
            if not splits.empty:
                s_df = splits.reset_index()
                s_df.columns = ['split_date', 'split_ratio']
                s_df['split_date'] = s_df['split_date'].dt.tz_localize(None)
                s_df['ticker'] = ticker
                s_df.to_parquet(CA_BATCH_DIR + f'{safe_name}.parquet', index=False)
                n_splits = len(s_df)

            new_checkpoint_rows.append({
                'ticker': ticker, 'status': 'success', 'rows_pulled': len(df),
                'n_splits': n_splits, 'error_message': '', 'timestamp': datetime.now().isoformat(),
            })
            success = True
            break  # do not retry after a genuine success

        except Exception as e:
            last_error = str(e)
            if attempt < MAX_ATTEMPTS:
                print(f"  [{ticker}] attempt {attempt} failed ({last_error}) — "
                      f"retrying after {RETRY_DELAY_SECONDS}s...")
                time.sleep(RETRY_DELAY_SECONDS)

    if not success:
        print(f"  [{ticker}] FAILED after {MAX_ATTEMPTS} attempts: {last_error}")
        new_checkpoint_rows.append({
            'ticker': ticker, 'status': 'failed', 'rows_pulled': 0,
            'n_splits': 0, 'error_message': last_error, 'timestamp': datetime.now().isoformat(),
        })

    # Periodic checkpoint flush: never lose more than CHECKPOINT_FLUSH_EVERY
    # tickers of progress if the runtime disconnects mid-run.
    if (i + 1) % CHECKPOINT_FLUSH_EVERY == 0 or (i + 1) == len(tickers_to_pull):
        print(f"Progress: {i + 1}/{len(tickers_to_pull)} tickers processed this run.")
        checkpoint = pd.concat([checkpoint, pd.DataFrame(new_checkpoint_rows)], ignore_index=True)
        checkpoint.to_csv(CHECKPOINT_PATH, index=False)
        new_checkpoint_rows = []

    time.sleep(DELAY_SECONDS)

# Final safety flush in case the last partial batch wasn't a multiple of
# CHECKPOINT_FLUSH_EVERY (already covered by the (i+1)==len(...) condition
# above, but kept here as a defensive no-op in case that branch is ever
# changed later).
if new_checkpoint_rows:
    checkpoint = pd.concat([checkpoint, pd.DataFrame(new_checkpoint_rows)], ignore_index=True)
    checkpoint.to_csv(CHECKPOINT_PATH, index=False)


# ------------------------------------------------------------
# Run summary.
#
# IMPORTANT: the checkpoint file is APPEND-ONLY — every run adds new
# rows rather than overwriting old ones. That means a ticker that
# failed in an earlier run and succeeded on a later retry will have
# MULTIPLE rows in the checkpoint (one 'failed', one 'success'). This
# is deliberately kept as a full audit trail of every attempt — but it
# means we must summarize by each ticker's LATEST status, not by raw
# row counts, or a ticker that eventually succeeded would still show
# up as "failed" in this summary just because of a stale earlier row.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("BULK PULL RUN COMPLETE (this session)")
print("=" * 70)

latest_status = (
    checkpoint.sort_values('timestamp')
    .groupby('ticker')
    .last()[['status', 'error_message']]
    .reset_index()
)

final_success = (latest_status['status'] == 'success').sum()
final_failed = (latest_status['status'] == 'failed').sum()
print(f"Total successful (by latest attempt per ticker): {final_success} / {len(pull_list_df)}")
print(f"Total still failed (by latest attempt per ticker): {final_failed}")

if final_failed > 0:
    print("\nTickers CURRENTLY failed (latest attempt only — investigate before treating the pull as complete):")
    print(latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']].to_string(index=False))

print("\nIf tickers remain unattempted or failed, just RE-RUN this cell — it will")
print("automatically skip anything already marked 'success' and retry the rest.")

# ======================================================================
# STAGE 3C-RETRY — for tickers STILL failing after simply re-running
# Stage 3C once (which should have already cleared transient failures
# like rate-limiting — confirmed SPICEJET.NS was one such false failure).
#
# This does something Stage 3C's plain retry CANNOT do: tries a .BO
# (BSE) fallback for tickers that fail under .NS. Many NSE-delisted
# companies remained listed on BSE, and Yahoo often retains that price
# history under the .BO suffix even after NSE delisting.
#
# Anything that STILL fails after the .BO fallback gets written to a
# separate review file — these are the ones we treat as probable real
# delistings/mergers UNLESS manually checked otherwise (e.g. the way we
# just confirmed SPICEJET.NS was a false failure via a web search).
# ======================================================================

import yfinance as yf
import pandas as pd
import os
import time
from datetime import datetime

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CA_BATCH_DIR = BASE + 'corporate_actions/batches/'

START = '2010-01-01'
END = pd.Timestamp.today().strftime('%Y-%m-%d')
DELAY_SECONDS = 0.75

checkpoint = pd.read_csv(CHECKPOINT_PATH)
latest_status = (
    checkpoint.sort_values('timestamp')
    .groupby('ticker')
    .last()[['status', 'error_message']]
    .reset_index()
)
currently_failed = latest_status[latest_status['status'] == 'failed']['ticker'].tolist()

print(f"{len(currently_failed)} tickers currently failed — attempting .BO fallback for each.\n")

new_checkpoint_rows = []
still_unresolved = []

for i, ns_ticker in enumerate(currently_failed):
    bo_ticker = ns_ticker.replace('.NS', '.BO')

    try:
        df = yf.download(bo_ticker, start=START, end=END, auto_adjust=False, progress=False)

        if df.empty:
            raise ValueError(f".BO fallback also returned empty for {bo_ticker}")

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # IMPORTANT: we save this under the ORIGINAL .NS-derived filename,
        # but tag which actual source ticker was used, so downstream
        # consumers of this file know it came from BSE, not NSE — this
        # matters if NSE and BSE prices ever diverge slightly (they can,
        # due to different liquidity/trading patterns on each exchange).
        df['ticker'] = ns_ticker              # keep original identity for joins
        df['source_ticker'] = bo_ticker        # but record the ACTUAL pull source
        df['exchange_source'] = 'BSE_fallback'
        df = df.reset_index()

        safe_name = ns_ticker.replace('.', '_')
        df.to_parquet(PRICE_BATCH_DIR + f'{safe_name}.parquet', index=False)

        print(f"  [{ns_ticker}] RECOVERED via BSE fallback ({bo_ticker}), {len(df)} rows.")
        new_checkpoint_rows.append({
            'ticker': ns_ticker, 'status': 'success', 'rows_pulled': len(df),
            'n_splits': 0, 'error_message': 'recovered_via_BSE_fallback',
            'timestamp': datetime.now().isoformat(),
        })

    except Exception as e:
        print(f"  [{ns_ticker}] still unresolved — .BO fallback also failed: {e}")
        still_unresolved.append({'ticker': ns_ticker, 'bo_attempt_error': str(e)})
        new_checkpoint_rows.append({
            'ticker': ns_ticker, 'status': 'failed', 'rows_pulled': 0,
            'n_splits': 0, 'error_message': f'NS and BO both failed: {e}',
            'timestamp': datetime.now().isoformat(),
        })

    time.sleep(DELAY_SECONDS)

checkpoint = pd.concat([checkpoint, pd.DataFrame(new_checkpoint_rows)], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# Save the still-unresolved list explicitly for manual review — these are
# our candidate "genuine delisting/merger, accept as a real gap" tickers,
# but should be EYEBALLED (a few spot-checked via web search, as we did
# for SpiceJet) rather than assumed correct without any check at all.
# FIX: pd.DataFrame([]) on an EMPTY list produces a DataFrame with ZERO
# columns (no headers at all), and saving that writes a genuinely EMPTY
# (0-byte) file — not an empty table. A later pd.read_csv() on that file
# crashes with "EmptyDataError: No columns to parse from file". This only
# surfaced once BSE fallback finally had zero tickers to retry (an edge
# case we hadn't hit before). Explicit columns fix it for the empty case too.
unresolved_df = pd.DataFrame(still_unresolved, columns=['ticker', 'bo_attempt_error'])
unresolved_path = BASE + 'validation/unresolved_tickers_for_manual_review.csv'
unresolved_df.to_csv(unresolved_path, index=False)

print("\n" + "=" * 70)
print(f"RECOVERED via BSE fallback: {len(currently_failed) - len(still_unresolved)}")
print(f"STILL UNRESOLVED (saved to {unresolved_path}): {len(still_unresolved)}")
print("=" * 70)
if still_unresolved:
    print("\nUnresolved tickers (spot-check a sample of these before accepting as real gaps):")
    print(unresolved_df.to_string(index=False))

# ======================================================================
# STAGE 3-QA — Check the SUCCESSFUL pulls, not just the failed ones.
#
# A ticker can return a non-empty DataFrame (no exception raised) and
# STILL be wrong: sparse date coverage, duplicate rows, all-NaN or
# garbage values, a schema that silently drifted. "No error" is not
# the same as "correct" — this script checks for the specific failure
# modes that wouldn't have shown up as a crash.
# ======================================================================

import pandas as pd
import numpy as np
import os
import glob

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
print(f"Found {len(batch_files)} successfully-pulled ticker files to QA.\n")

qa_rows = []

for fpath in batch_files:
    ticker = os.path.basename(fpath).replace('_parquet', '').replace('.parquet', '').replace('_', '.', 1)
    # NOTE: the replace('_', '.', 1) above assumes tickers have exactly one
    # underscore standing in for the original '.' (e.g. RELIANCE_NS ->
    # RELIANCE.NS). This holds for plain '.NS'/'.BO' tickers but would
    # break if a ticker itself ever contained an underscore — worth
    # re-checking if that ever comes up, not assumed silently correct here.

    df = pd.read_parquet(fpath)

    n_rows = len(df)
    n_duplicate_dates = df['Date'].duplicated().sum() if 'Date' in df.columns else np.nan
    date_min = df['Date'].min() if 'Date' in df.columns and n_rows > 0 else None
    date_max = df['Date'].max() if 'Date' in df.columns and n_rows > 0 else None

    # % NaN in each core OHLCV column
    core_cols = [c for c in ['Open', 'High', 'Low', 'Close', 'Volume'] if c in df.columns]
    pct_nan = {c: df[c].isna().mean() * 100 for c in core_cols}

    # Sanity: any non-positive Close or negative Volume? Should never happen
    # in legitimate price data.
    n_nonpositive_close = (df['Close'] <= 0).sum() if 'Close' in df.columns else np.nan
    n_negative_volume = (df['Volume'] < 0).sum() if 'Volume' in df.columns else np.nan

    # Expected trading days is roughly 252/year. A HUGE gap between actual
    # row count and (date_max - date_min) in calendar days, adjusted for
    # weekends/holidays, suggests missing chunks of history rather than
    # a company that simply listed late.
    if date_min is not None and date_max is not None:
        calendar_days_span = (date_max - date_min).days
        expected_trading_days_approx = calendar_days_span * (252 / 365)
        coverage_ratio = n_rows / expected_trading_days_approx if expected_trading_days_approx > 0 else np.nan
    else:
        coverage_ratio = np.nan

    qa_rows.append({
        'ticker': ticker,
        'n_rows': n_rows,
        'date_min': date_min,
        'date_max': date_max,
        'n_duplicate_dates': n_duplicate_dates,
        'pct_nan_close': pct_nan.get('Close', np.nan),
        'pct_nan_volume': pct_nan.get('Volume', np.nan),
        'n_nonpositive_close': n_nonpositive_close,
        'n_negative_volume': n_negative_volume,
        'coverage_ratio': coverage_ratio,  # ~1.0 = dense, well below 1.0 = gappy
    })

qa_df = pd.DataFrame(qa_rows)
qa_path = BASE + 'validation/successful_pull_qa_report.csv'
qa_df.to_csv(qa_path, index=False)

print("=" * 70)
print("QA SUMMARY")
print("=" * 70)

print(f"\nTotal tickers QA'd: {len(qa_df)}")

dupes_found = qa_df[qa_df['n_duplicate_dates'] > 0]
print(f"\nTickers with duplicate dates within their own file: {len(dupes_found)}")
if not dupes_found.empty:
    print(dupes_found[['ticker', 'n_duplicate_dates']].to_string(index=False))

bad_close = qa_df[qa_df['n_nonpositive_close'] > 0]
print(f"\nTickers with non-positive Close values (should be ZERO, ever): {len(bad_close)}")
if not bad_close.empty:
    print(bad_close[['ticker', 'n_nonpositive_close']].to_string(index=False))

bad_volume = qa_df[qa_df['n_negative_volume'] > 0]
print(f"\nTickers with negative Volume values (should be ZERO, ever): {len(bad_volume)}")
if not bad_volume.empty:
    print(bad_volume[['ticker', 'n_negative_volume']].to_string(index=False))

high_nan = qa_df[qa_df['pct_nan_close'] > 5]
print(f"\nTickers with >5% NaN in Close price: {len(high_nan)}")
if not high_nan.empty:
    print(high_nan[['ticker', 'pct_nan_close']].to_string(index=False))

sparse = qa_df[qa_df['coverage_ratio'] < 0.85]
print(f"\nTickers with sparse coverage (<85% of expected trading days present — "
      f"possible large gaps in the middle of their history, not just a late listing): {len(sparse)}")
if not sparse.empty:
    print(sparse[['ticker', 'n_rows', 'date_min', 'date_max', 'coverage_ratio']]
          .sort_values('coverage_ratio').to_string(index=False))

print(f"\nFull QA report saved to: {qa_path}")
print("\nNOTE: a LOW coverage_ratio is expected and FINE for a company that")
print("listed after 2010 (it just has less history, not gappy history) — check")
print("date_min for those before treating them as a real problem. The concerning")
print("case is a coverage_ratio well below 1.0 for a stock with an early date_min,")
print("which points to actual missing chunks mid-history, not a late listing.")

# ======================================================================
# STAGE 3-QA2 — Cross-validate price history against the PIT universe.
#
# WHY THIS CHECK EXISTS (separate from the earlier QA pass):
# The earlier QA only checked each ticker's data against ITSELF (is its
# own date range internally dense). It could NOT catch a ticker whose
# entire history silently starts too late — e.g. if a company renamed
# and yfinance treats the new ticker as a fresh listing rather than
# continuing the old symbol's price history. That would "succeed" with
# no error, and look internally fine, while actually missing years of
# real, needed history.
#
# This check instead compares each ticker's ACTUAL price start date
# against the TRUE first-appearance date recorded for its ISIN in our
# authoritative canonical PIT universe file (built from Bloomberg
# Sheet4) — this is the one thing we have that tells us independently
# when a company should have data, rather than trusting the price file
# to judge itself.
# ======================================================================

import pandas as pd
import numpy as np
import glob
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
UNIVERSE_PATH = BASE + 'universe/canonical_membership_semiannual.parquet'

canonical = pd.read_parquet(UNIVERSE_PATH)

# Same "#N/A" handling as Stage 3B — must stay consistent with how the
# pull list was originally derived, or this check would compare against
# the wrong reference for the excluded/edge-case ISINs.
is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')
valid_membership = canonical[~is_missing_row].copy()

# For each ISIN: the ticker we actually pulled (latest valid symbol) AND
# the TRUE first appearance date across its ENTIRE history under that
# ISIN — not just the most recent symbol era, but from whenever this
# ISIN first shows up as a constituent at all.
isin_summary = (
    valid_membership.sort_values('snap_date')
    .groupby('isin')
    .agg(exch_symbol=('exch_symbol', 'last'),
         true_first_appearance=('snap_date', 'min'))
    .reset_index()
)
isin_summary['ticker'] = isin_summary['exch_symbol'].str.strip() + '.NS'

# Load actual date_min for every successfully pulled ticker.
batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
price_start_rows = []
for fpath in batch_files:
    ticker = os.path.basename(fpath).replace('.parquet', '').replace('_', '.', 1)
    df = pd.read_parquet(fpath, columns=['Date'])
    if df.empty:
        continue
    price_start_rows.append({'ticker': ticker, 'actual_price_start': df['Date'].min()})

price_starts = pd.DataFrame(price_start_rows)

# The actual cross-validation.
merged = price_starts.merge(
    isin_summary[['ticker', 'true_first_appearance']],
    on='ticker', how='left'
)
merged['gap_days'] = (merged['actual_price_start'] - merged['true_first_appearance']).dt.days

# A ticker's price data STARTING LATER than its true first appearance as
# a constituent is the concerning case (missing real history). A price
# series starting EARLIER than PIT-constituent-entry is fine and
# expected (a company can trade for years before entering the Nifty 500).
FLAG_THRESHOLD_DAYS = 180  # more than ~6 months late suggests a real gap, not vendor noise

no_reference = merged[merged['true_first_appearance'].isna()]
flagged = merged[merged['gap_days'] > FLAG_THRESHOLD_DAYS].sort_values('gap_days', ascending=False)

print("=" * 70)
print("CROSS-VALIDATION: price start date vs. true PIT first-appearance date")
print("=" * 70)

print(f"\nTotal successfully-pulled tickers checked: {len(merged)}")
print(f"Tickers with NO matching ISIN reference found (investigate separately "
      f"— likely a join mismatch, not a real problem, but worth confirming): {len(no_reference)}")
if not no_reference.empty:
    print(no_reference[['ticker']].to_string(index=False))

print(f"\nTickers whose price data starts >{FLAG_THRESHOLD_DAYS} days AFTER their "
      f"true first appearance as a constituent (SUSPECTED SILENT GAP): {len(flagged)}")
if not flagged.empty:
    print(flagged[['ticker', 'true_first_appearance', 'actual_price_start', 'gap_days']]
          .to_string(index=False))
else:
    print("None found — no evidence of the 'renamed ticker silently truncates history' failure mode.")

qa2_path = BASE + 'validation/pit_crossvalidation_report.csv'
merged.to_csv(qa2_path, index=False)
print(f"\nFull cross-validation report saved to: {qa2_path}")

# ======================================================================
# MANUAL FIX — HINDMOTORS.NS "recovered" via BSE fallback with only 1
# row (a phantom quote, not real history). Reclassify as failed so it's
# not mistaken for usable data.
# ======================================================================
import os
import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

bad_ticker = 'HINDMOTORS.NS'

# Delete the useless 1-row file so it can't be mistaken for real data later
bad_file = PRICE_BATCH_DIR + bad_ticker.replace('.', '_') + '.parquet'
if os.path.exists(bad_file):
    os.remove(bad_file)

    print(f"Removed {bad_file} (only 1 row — not usable).")

checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_row = pd.DataFrame([{
    'ticker': bad_ticker, 'status': 'failed', 'rows_pulled': 0,
    'n_splits': 0, 'error_message': 'recovered_via_BSE_but_only_1_row_treated_as_unusable',
    'timestamp': pd.Timestamp.now().isoformat(),
}])
checkpoint = pd.concat([checkpoint, new_row], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)
print("Checkpoint updated — HINDMOTORS.NS now correctly shows as failed/unusable.")

# ======================================================================
# RECLASSIFICATION — the PIT cross-validation above (Stage 3-QA2) can
# flag tickers with real gaps (phantom single-row quotes, or genuine
# but incomplete history starting well after true PIT entry). Move any
# such tickers from 'success' back to 'failed' so they don't silently
# pollute the panel.
# ======================================================================
import pandas as pd
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CV_REPORT_PATH = BASE + 'validation/pit_crossvalidation_report.csv'
UNRESOLVED_PATH = BASE + 'validation/unresolved_tickers_for_manual_review.csv'

FLAG_THRESHOLD_DAYS = 180

cv = pd.read_csv(CV_REPORT_PATH)
flagged = cv[cv['gap_days'] > FLAG_THRESHOLD_DAYS].copy()

print(f"Reclassifying {len(flagged)} tickers from 'success' to 'failed' — "
      f"they have real gaps against the authoritative PIT universe.\n")

checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_rows = []

for _, row in flagged.iterrows():
    ticker = row['ticker']
    gap = row['gap_days']

    # Distinguish the two patterns for the log, since they have different
    # implications: Pattern A (phantom quote) vs Pattern B (real but late data).
    fpath = PRICE_BATCH_DIR + ticker.replace('.', '_') + '.parquet'
    if os.path.exists(fpath):
        df = pd.read_parquet(fpath)
        n_rows = len(df)
    else:
        n_rows = 0

    if n_rows <= 2:
        reason = f'phantom_single_row_quote_gap_{int(gap)}_days'
        if os.path.exists(fpath):
            os.remove(fpath)
            print(f"  [{ticker}] Removed phantom {n_rows}-row file. Gap: {int(gap)} days.")
    else:
        reason = f'real_data_but_starts_{int(gap)}_days_after_true_PIT_entry'
        print(f"  [{ticker}] Flagged as incomplete (kept file for reference, "
              f"{n_rows} rows, but starts {int(gap)} days late).")

    new_rows.append({
        'ticker': ticker, 'status': 'failed', 'rows_pulled': 0 if n_rows <= 2 else n_rows,
        'n_splits': 0, 'error_message': reason, 'timestamp': pd.Timestamp.now().isoformat(),
    })

checkpoint = pd.concat([checkpoint, pd.DataFrame(new_rows)], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# FIX: this used to READ the existing unresolved CSV and MERGE new rows
# into it — the exact hand-merge pattern we banned everywhere else in
# this pipeline after it drifted stale once already (see build_log.md).
# It also crashed outright the one time the file happened to be empty
# (0 bytes, from a separate bug in the BSE-fallback step, fixed above) —
# pd.read_csv() cannot parse a truly empty file. Fixed to do what every
# other stage in this pipeline does: regenerate the unresolved list
# FRESH from checkpoint (the actual source of truth), never merge.
latest_status_final = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()
combined_unresolved = latest_status_final[latest_status_final['status'] == 'failed'][['ticker', 'error_message']]
combined_unresolved.to_csv(UNRESOLVED_PATH, index=False)

print(f"\nConsolidated unresolved list now has {len(combined_unresolved)} tickers total "
      f"(saved to {UNRESOLVED_PATH}).")

# Final true count
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last()['status']
true_success_count = (latest_status == 'success').sum()
print(f"\nTRUE current successful ticker count after reclassification: {true_success_count}")

import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'
UNRESOLVED_PATH = BASE + 'validation/unresolved_tickers_for_manual_review.csv'
PULL_LIST_PATH = BASE + 'universe/pull_ticker_list_v1.csv'

checkpoint = pd.read_csv(CHECKPOINT_PATH)
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last()['status']

unresolved_file = pd.read_csv(UNRESOLVED_PATH)
pull_list = pd.read_csv(PULL_LIST_PATH)

print("="*70)
print("RECONCILIATION: checkpoint (source of truth) vs unresolved CSV file")
print("="*70)

# 1. Does every ticker in the pull list have a checkpoint entry at all?
missing_from_checkpoint = set(pull_list['yf_ticker']) - set(latest_status.index)
print(f"\nTickers in pull list with NO checkpoint entry at all: {len(missing_from_checkpoint)}")
if missing_from_checkpoint:
    print(sorted(missing_from_checkpoint))

# 2. Does the unresolved CSV contain any ticker that checkpoint says is ACTUALLY successful?
stale_in_unresolved = [
    t for t in unresolved_file['ticker']
    if t in latest_status.index and latest_status[t] == 'success'
]
print(f"\nTickers in unresolved_tickers_for_manual_review.csv that checkpoint "
      f"says are ACTUALLY successful (stale entries): {len(stale_in_unresolved)}")
if stale_in_unresolved:
    print(sorted(stale_in_unresolved))

# 3. Does checkpoint have 'failed' tickers that are MISSING from the unresolved CSV?
checkpoint_failed = set(latest_status[latest_status == 'failed'].index)
unresolved_set = set(unresolved_file['ticker'])
missing_from_unresolved = checkpoint_failed - unresolved_set
print(f"\nTickers checkpoint says are 'failed' but are MISSING from the unresolved CSV: "
      f"{len(missing_from_unresolved)}")
if missing_from_unresolved:
    print(sorted(missing_from_unresolved))

# 4. The TRUE, trustworthy numbers, straight from checkpoint only.
true_success = (latest_status == 'success').sum()
true_failed = (latest_status == 'failed').sum()
print(f"\n--- TRUE STATE (from checkpoint only) ---")
print(f"Success: {true_success}")
print(f"Failed: {true_failed}")
print(f"Total accounted for: {true_success + true_failed} (pull list has {len(pull_list)})")

import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
checkpoint = pd.read_csv(BASE + 'raw/prices/_pull_progress.csv')
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()

failed_now = latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']]
failed_now.to_csv(BASE + 'validation/unresolved_tickers_for_manual_review.csv', index=False)
print(f"Regenerated unresolved list from checkpoint: {len(failed_now)} tickers.")

# ======================================================================
# Generate the Bloomberg pull list for the 116 unresolved tickers.
#
# IMPORTANT: uses bbg_id (e.g. "RANBAXY IS Equity"), NOT the yfinance
# .NS ticker — Bloomberg won't recognize .NS-style strings, and bbg_id
# is the identifier we already confirmed works (it's what Sheet4 itself
# was built from).
# ======================================================================

import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'

canonical = pd.read_parquet(BASE + 'universe/canonical_membership_semiannual.parquet')
checkpoint = pd.read_csv(BASE + 'raw/prices/_pull_progress.csv')

latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()
unresolved_tickers = latest_status[latest_status['status'] == 'failed']['ticker'].tolist()

print(f"{len(unresolved_tickers)} unresolved tickers to build a Bloomberg pull list for.")

# Rebuild the ISIN -> bbg_id mapping the same way we built ISIN -> yf_ticker
# in Stage 3B, so this stays consistent with how the original pull list
# was derived.
is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')
valid_membership = canonical[~is_missing_row]

isin_map = (
    valid_membership.sort_values('snap_date')
    .groupby('isin')
    .last()[['exch_symbol', 'bbg_id', 'name']]
    .reset_index()
)
isin_map['yf_ticker'] = isin_map['exch_symbol'].str.strip() + '.NS'

bloomberg_pull_list = isin_map[isin_map['yf_ticker'].isin(unresolved_tickers)][
    ['isin', 'yf_ticker', 'bbg_id', 'name']
].sort_values('bbg_id')

print(f"\nMatched {len(bloomberg_pull_list)} of {len(unresolved_tickers)} unresolved "
      f"tickers to a bbg_id.")
if len(bloomberg_pull_list) != len(unresolved_tickers):
    missing = set(unresolved_tickers) - set(bloomberg_pull_list['yf_ticker'])
    print(f"WARNING: {len(missing)} unresolved tickers had no bbg_id match — investigate: {missing}")

out_path = BASE + 'validation/bloomberg_pull_list_for_unresolved.csv'
bloomberg_pull_list.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")
print("\nJust the bbg_id column, ready to copy into Excel:\n")
print(bloomberg_pull_list['bbg_id'].to_string(index=False))

# ======================================================================
# BLOOMBERG MERGE — parse the 116-ticker Bloomberg export, truncate
# trailing NaN rows to each ticker's real last-valid date, write
# per-ticker files matching the existing yfinance batch convention,
# and update the checkpoint.
#
# CONFIRMED WITH BLOOMBERG: this data is already split-adjusted.
# No custom adjustment step needed (unlike the earlier yfinance saga).
# ======================================================================

import pandas as pd
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'

# ------------------------------------------------------------
# STEP 1: Load the Bloomberg export. Upload it if prompted.
# ------------------------------------------------------------
from google.colab import files
print("Upload bloomberg_pull_list_for_unresolved.xlsx")
uploaded = files.upload()
xlsx_path = list(uploaded.keys())[0]

pull_list = pd.read_excel(xlsx_path, sheet_name='bloomberg_pull_list_for_unresol')
bbgid_to_ticker = dict(zip(pull_list['bbg_id'], pull_list['yf_ticker']))

df = pd.read_excel(xlsx_path, sheet_name='bloomberg_pull_list_for_unr (2)', header=[0, 1])
new_cols = list(df.columns)
new_cols[0] = ('DATE', '')  # rename the shared date column for a clean stack()
df.columns = pd.MultiIndex.from_tuples(new_cols)

df_long = df.set_index(('DATE', '')).stack(level=0, future_stack=True).reset_index()
df_long.columns = ['Date', 'bbg_id', 'Open', 'High', 'Low', 'Close', 'Volume']

# ------------------------------------------------------------
# STEP 2: Map bbg_id -> our canonical yf_ticker identity, so the WHOLE
# panel (yfinance + Bloomberg sourced) uses one consistent identity
# scheme regardless of which vendor actually supplied the data.
# ------------------------------------------------------------
df_long['ticker'] = df_long['bbg_id'].map(bbgid_to_ticker)

unmapped = df_long[df_long['ticker'].isna()]['bbg_id'].unique()
if len(unmapped) > 0:
    raise ValueError(f"{len(unmapped)} bbg_ids failed to map to a ticker — "
                      f"stop and investigate: {unmapped}")
print(f"All {df_long['bbg_id'].nunique()} bbg_ids mapped cleanly to tickers.")

df_long['source'] = 'bloomberg'
df_long = df_long.drop(columns=['bbg_id'])

# ------------------------------------------------------------
# STEP 3: Truncate trailing NaN rows per ticker.
#
# WHY: Bloomberg gave every security a row for every date in the shared
# grid, even long after it stopped trading — those rows are all-NaN,
# not real data. Left in place, they'd silently sit in the panel
# implying the security "exists" all the way to today. We truncate each
# ticker to its own true last-valid-Close date, and LOG the truncation
# explicitly rather than just dropping rows invisibly.
# ------------------------------------------------------------
truncation_log = []
cleaned_frames = []

for ticker, sub in df_long.groupby('ticker'):
    sub = sub.sort_values('Date').reset_index(drop=True)
    valid = sub.dropna(subset=['Close'])

    if valid.empty:
        truncation_log.append({
            'ticker': ticker, 'status': 'NO_VALID_DATA',
            'last_valid_date': None, 'rows_dropped': len(sub),
        })
        continue  # do not write a file for a fully-empty ticker

    last_valid_date = valid['Date'].max()
    kept = sub[sub['Date'] <= last_valid_date].copy()
    dropped_count = len(sub) - len(kept)

    truncation_log.append({
        'ticker': ticker, 'status': 'truncated' if dropped_count > 0 else 'no_truncation_needed',
        'last_valid_date': last_valid_date, 'rows_dropped': dropped_count,
    })
    cleaned_frames.append(kept)

truncation_df = pd.DataFrame(truncation_log)
truncation_path = BASE + 'validation/bloomberg_truncation_log.csv'
truncation_df.to_csv(truncation_path, index=False)

no_data_tickers = truncation_df[truncation_df['status'] == 'NO_VALID_DATA']['ticker'].tolist()
print(f"\n{len(no_data_tickers)} ticker(s) had NO valid data at all in the Bloomberg pull "
      f"(genuinely still unresolved): {no_data_tickers}")
print(f"Truncation log saved to: {truncation_path}")

# ------------------------------------------------------------
# STEP 4: Write one file per ticker, matching the EXACT filename
# convention already used for the yfinance batches.
# ------------------------------------------------------------
n_written = 0
for frame in cleaned_frames:
    ticker = frame['ticker'].iloc[0]
    safe_name = ticker.replace('.', '_')
    frame.to_parquet(PRICE_BATCH_DIR + f'{safe_name}.parquet', index=False)
    n_written += 1

print(f"\nWrote {n_written} Bloomberg-sourced ticker files to {PRICE_BATCH_DIR}")

# ------------------------------------------------------------
# STEP 5: Update the checkpoint. Tickers with real data -> success.
# The genuinely-empty ticker(s) stay 'failed', with a clear reason.
# ------------------------------------------------------------
checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_rows = []

for _, row in truncation_df.iterrows():
    if row['status'] == 'NO_VALID_DATA':
        new_rows.append({
            'ticker': row['ticker'], 'status': 'failed', 'rows_pulled': 0,
            'n_splits': 0, 'error_message': 'bloomberg_pull_also_returned_no_valid_data',
            'timestamp': pd.Timestamp.now().isoformat(),
        })
    else:
        n_rows = pd.read_parquet(PRICE_BATCH_DIR + row['ticker'].replace('.', '_') + '.parquet').shape[0]
        new_rows.append({
            'ticker': row['ticker'], 'status': 'success', 'rows_pulled': n_rows,
            'n_splits': 0, 'error_message': 'recovered_via_bloomberg_terminal_pull',
            'timestamp': pd.Timestamp.now().isoformat(),
        })

checkpoint = pd.concat([checkpoint, pd.DataFrame(new_rows)], ignore_index=True)
checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# ------------------------------------------------------------
# STEP 6: Regenerate the unresolved list FRESH from checkpoint
# (never hand-merge this file again — lesson learned earlier this session).
# ------------------------------------------------------------
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()
still_failed = latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']]
still_failed.to_csv(BASE + 'validation/unresolved_tickers_for_manual_review.csv', index=False)

true_success = (latest_status['status'] == 'success').sum()
true_failed = (latest_status['status'] == 'failed').sum()

print(f"\n" + "=" * 70)
print("FINAL STATE AFTER BLOOMBERG MERGE")
print("=" * 70)
print(f"Success: {true_success}")
print(f"Still failed: {true_failed}")
print(f"Total: {true_success + true_failed} (should be 1071)")

# ======================================================================
# SPARSE-TICKER RECLASSIFICATION — generalized, not hardcoded to one name.
#
# NaN percentage alone is misleading for usability: 97% NaN on a 226-row
# file leaves ~6 usable days (unusable); 5% NaN on a 4000-row file leaves
# ~3800 (fine). This checks the ABSOLUTE valid-row count for EVERY
# currently-successful ticker against the longest planned feature window
# (realized_vol_120), so any ticker too sparse to support that feature
# gets caught and reclassified automatically — not just DENABANK.NS
# (found this session with only 6 valid Close values), but any similarly
# sparse ticker a future re-run on refreshed data might turn up.
#
# MUST run here: after the Bloomberg merge (some sparse tickers, like
# DENABANK.NS, are Bloomberg-sourced and don't exist as batch files
# until that step completes) and before Stage 3D consolidation (or the
# bad data gets baked into the final panel).
# ======================================================================

import pandas as pd
import glob
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'
CHECKPOINT_PATH = BASE + 'raw/prices/_pull_progress.csv'

MIN_VALID_ROWS = 120  # longest planned feature is realized_vol_120

batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
sparse_found = []

for fpath in batch_files:
    df = pd.read_parquet(fpath, columns=['ticker', 'Close'])
    n_valid = df['Close'].notna().sum()
    if n_valid < MIN_VALID_ROWS:
        ticker = df['ticker'].iloc[0]
        sparse_found.append({'ticker': ticker, 'n_valid_close': n_valid, 'fpath': fpath})

print(f"Checked {len(batch_files)} batch files against the {MIN_VALID_ROWS}-day minimum.")
print(f"Found {len(sparse_found)} ticker(s) below threshold:")

checkpoint = pd.read_csv(CHECKPOINT_PATH)
new_rows = []

for item in sparse_found:
    ticker = item['ticker']
    n_valid = item['n_valid_close']
    print(f"  [{ticker}] only {n_valid} valid Close values — reclassifying as failed.")

    os.remove(item['fpath'])
    new_rows.append({
        'ticker': ticker, 'status': 'failed', 'rows_pulled': 0, 'n_splits': 0,
        'error_message': f'only_{n_valid}_valid_close_values_below_{MIN_VALID_ROWS}_day_minimum',
        'timestamp': pd.Timestamp.now().isoformat(),
    })

if new_rows:
    checkpoint = pd.concat([checkpoint, pd.DataFrame(new_rows)], ignore_index=True)
    checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# Regenerate unresolved list fresh from checkpoint (never hand-merge)
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last().reset_index()
still_failed = latest_status[latest_status['status'] == 'failed'][['ticker', 'error_message']]
still_failed.to_csv(BASE + 'validation/unresolved_tickers_for_manual_review.csv', index=False)

true_success = (latest_status['status'] == 'success').sum()
true_failed = (latest_status['status'] == 'failed').sum()
print(f"\nUpdated true state: {true_success} success, {true_failed} failed, "
      f"{true_success + true_failed} total.")
print(f"All failed tickers: {sorted(still_failed['ticker'].tolist())}")

# ======================================================================
# STAGE 3D — Consolidate all per-ticker batch files (yfinance + Bloomberg)
# into one final raw price panel, with schema harmonized across both
# sources, and a final QA pass on the fully merged result.
# ======================================================================

import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

batch_files = glob.glob(PRICE_BATCH_DIR + '*.parquet')
print(f"Found {len(batch_files)} per-ticker batch files to consolidate.")

# ------------------------------------------------------------
# Schema harmonization: the 955 yfinance-sourced files carry an
# 'Adj Close' column (dividend-adjusted, NOT used for our price-return
# work) and no 'source' column. The 115 Bloomberg-sourced files carry
# 'source' but no 'Adj Close'. We standardize both onto one final schema
# so concatenation doesn't produce a sparse, misaligned table -- the
# exact bug we hit early on with yfinance's own MultiIndex columns.
# ------------------------------------------------------------
FINAL_COLS = ['Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
              'adj_close_dividend_yfinance_unused', 'source']

def harmonize(df, default_source):
    df = df.copy()
    if 'source' not in df.columns:
        df['source'] = default_source
    if 'Adj Close' in df.columns:
        df = df.rename(columns={'Adj Close': 'adj_close_dividend_yfinance_unused'})
    else:
        df['adj_close_dividend_yfinance_unused'] = np.nan
    return df[FINAL_COLS]

frames = []
for fpath in batch_files:
    df = pd.read_parquet(fpath)
    # yfinance batch files never had a 'source' column written; Bloomberg
    # ones always do (added explicitly in the merge script). This default
    # correctly back-fills the 955 older files without needing to touch them.
    frames.append(harmonize(df, default_source='yfinance'))

full_panel = pd.concat(frames, ignore_index=True)
print(f"\nConsolidated panel shape: {full_panel.shape}")
print(f"Source breakdown:\n{full_panel.groupby('source')['ticker'].nunique()}")

# ------------------------------------------------------------
# Final QA pass on the FULLY MERGED result -- same checks as Stage 3-QA,
# run once more here because merging can itself introduce problems
# (e.g. a ticker accidentally represented in BOTH source files, which
# would show as duplicate (ticker, Date) rows here even if each
# individual file was clean on its own).
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL QA ON CONSOLIDATED PANEL")
print("=" * 70)

dupes = full_panel.duplicated(subset=['ticker', 'Date']).sum()
print(f"\nDuplicate (ticker, Date) rows across the WHOLE consolidated panel: {dupes}")
if dupes > 0:
    dupe_tickers = full_panel[full_panel.duplicated(subset=['ticker','Date'], keep=False)]['ticker'].unique()
    print(f"Affected tickers (investigate before proceeding): {dupe_tickers}")

n_nonpositive_close = (full_panel['Close'] <= 0).sum()
n_negative_volume = (full_panel['Volume'] < 0).sum()
print(f"Non-positive Close values: {n_nonpositive_close}")
print(f"Negative Volume values: {n_negative_volume}")

n_unique_tickers = full_panel['ticker'].nunique()
print(f"\nTotal unique tickers in consolidated panel: {n_unique_tickers} (expect 1070)")

assert dupes == 0, "STOP: duplicate (ticker, Date) rows found -- do not proceed until resolved"
assert n_nonpositive_close == 0, "STOP: non-positive Close values found"
assert n_negative_volume == 0, "STOP: negative Volume values found"
print("\n[PASS] Consolidated panel is clean: no duplicates, no invalid values.")

# ------------------------------------------------------------
# Save, timestamped and versioned -- never overwritten, consistent with
# every other raw file in this project.
# ------------------------------------------------------------
today_str = datetime.now().strftime('%Y-%m-%d')
out_path = BASE + f'raw/prices/raw_price_panel_consolidated_v1_{today_str}.parquet'
full_panel.to_parquet(out_path, index=False)
print(f"\nSaved consolidated raw price panel to: {out_path}")
print(f"This is the FINAL raw price panel for Stage 3 -- ready for the daily")
print(f"PIT-flag join and feature computation in the next stage.")

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HINDMOTORS.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2026-07-15)')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder structure created under: /content/drive/MyDrive/moonshot_compounding/
 - /content/drive/MyDrive/moonshot_compounding/raw/universe
 - /content/drive/MyDrive/moonshot_compounding/raw/prices
 - /content/drive/MyDrive/moonshot_compounding/corporate_actions
 - /content/drive/MyDrive/moonshot_compounding/adjusted
 - /content/drive/MyDrive/moonshot_compounding/universe
 - /content/drive/MyDrive/moonshot_compounding/features
 - /content/drive/MyDrive/moonshot_compounding/validation
Found existing canonical universe file at /content/drive/MyDrive/moonshot_compounding/universe/canonical_membership_semiannual.parquet — loading it.

Canonical universe loaded: 16507 rows, 1076 unique ISINs, 33 snapshot dates.

Derived 1071 pullable tickers from Sheet4.
Excluded 5 ISINs with NO valid symbol on any snapshot:
        isin                        name                   

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HINDMOTORS.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2026-07-15)')


  [HINDMOTORS.NS] attempt 2 failed (yfinance returned an empty DataFrame (ticker may be delisted, merged, or an invalid/incorrect symbol)) — retrying after 5s...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HINDMOTORS.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2026-07-15)')


  [HINDMOTORS.NS] FAILED after 3 attempts: yfinance returned an empty DataFrame (ticker may be delisted, merged, or an invalid/incorrect symbol)
Progress: 2/2 tickers processed this run.


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HINDMOTORS.BO']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2026-07-15)')



BULK PULL RUN COMPLETE (this session)
Total successful (by latest attempt per ticker): 1070 / 1071
Total still failed (by latest attempt per ticker): 1

Tickers CURRENTLY failed (latest attempt only — investigate before treating the pull as complete):
       ticker                                                                                         error_message
HINDMOTORS.NS yfinance returned an empty DataFrame (ticker may be delisted, merged, or an invalid/incorrect symbol)

If tickers remain unattempted or failed, just RE-RUN this cell — it will
automatically skip anything already marked 'success' and retry the rest.
1 tickers currently failed — attempting .BO fallback for each.

  [HINDMOTORS.NS] still unresolved — .BO fallback also failed: .BO fallback also returned empty for HINDMOTORS.BO

RECOVERED via BSE fallback: 0
STILL UNRESOLVED (saved to /content/drive/MyDrive/moonshot_compounding/validation/unresolved_tickers_for_manual_review.csv): 1

Unresolved tickers (spot-check 

Saving bloomberg_pull_list_for_unresolved.xlsx to bloomberg_pull_list_for_unresolved.xlsx
All 116 bbg_ids mapped cleanly to tickers.

1 ticker(s) had NO valid data at all in the Bloomberg pull (genuinely still unresolved): ['MIRCELECTR.NS']
Truncation log saved to: /content/drive/MyDrive/moonshot_compounding/validation/bloomberg_truncation_log.csv

Wrote 115 Bloomberg-sourced ticker files to /content/drive/MyDrive/moonshot_compounding/raw/prices/batches/

FINAL STATE AFTER BLOOMBERG MERGE
Success: 1070
Still failed: 1
Total: 1071 (should be 1071)
Checked 1070 batch files against the 120-day minimum.
Found 1 ticker(s) below threshold:
  [DENABANK.NS] only 6 valid Close values — reclassifying as failed.

Updated true state: 1069 success, 2 failed, 1071 total.
All failed tickers: ['DENABANK.NS', 'MIRCELECTR.NS']
Found 1069 per-ticker batch files to consolidate.

Consolidated panel shape: (3332879, 9)
Source breakdown:
source
bloomberg    114
yfinance     955
Name: ticker, dtype: int64

FI

In [4]:
# ======================================================================
# STAGE 3 — FINAL CONFIDENCE CHECK (read-only, no mutations)
#
# Run this AFTER the main production pipeline cell completes. This cell
# does not modify checkpoint, batch files, or the consolidated panel —
# it only READS and REPORTS. Safe to re-run any number of times.
#
# Covers, in order:
#   A. Schema conformance of the consolidated panel
#   B. Source column integrity
#   C. Close vs adj_close_dividend_yfinance_unused distinctness
#   D. PIT flag absence (confirmed expected, not a bug)
#   E. Ticker count reconciliation (panel vs checkpoint vs universe)
#   F. Independent re-check: duplicates, non-positive close, negative volume
#   G. Sparse-ticker re-check (confirm NONE remain below the 120-day minimum)
#   H. NaN-rate summary (informational — these are accepted, not blocking)
#   I. PIT cross-validation re-check (confirm 0 silent-gap tickers remain)
#   J. Named-ticker spot check on the two known permanent gaps + the
#      HINDMOTORS.NS recovery, to confirm the story the summary told us
#      is actually true in the data, not just in the printed log
# ======================================================================

import pandas as pd
import numpy as np
import glob
import os

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

issues_found = []  # collect problems here; print a clear pass/fail banner at the end

# ------------------------------------------------------------
# Load the most recent consolidated panel
# ------------------------------------------------------------
panel_files = sorted(glob.glob(BASE + 'raw/prices/raw_price_panel_consolidated_v1_*.parquet'))
assert panel_files, "No consolidated panel file found — did Stage 3D run?"
latest_panel_path = panel_files[-1]
print(f"Checking: {latest_panel_path}\n")
df = pd.read_parquet(latest_panel_path)

checkpoint = pd.read_csv(BASE + 'raw/prices/_pull_progress.csv')
latest_status = checkpoint.sort_values('timestamp').groupby('ticker').last()['status']
pull_list = pd.read_csv(BASE + 'universe/pull_ticker_list_v1.csv')

# ------------------------------------------------------------
# A. SCHEMA CONFORMANCE
# ------------------------------------------------------------
print("=" * 70)
print("A. SCHEMA CONFORMANCE")
print("=" * 70)
expected_cols = {'Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
                  'adj_close_dividend_yfinance_unused', 'source'}
actual_cols = set(df.columns)
missing = expected_cols - actual_cols
extra = actual_cols - expected_cols
print(f"Missing expected columns: {missing}")
print(f"Unexpected extra columns: {extra}")
if missing or extra:
    issues_found.append(f"Schema mismatch: missing={missing}, extra={extra}")
else:
    print("[PASS]")

# ------------------------------------------------------------
# B. SOURCE COLUMN INTEGRITY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("B. SOURCE COLUMN INTEGRITY")
print("=" * 70)
unique_sources = set(df['source'].unique())
n_null_source = df['source'].isna().sum()
print(f"Unique source values: {unique_sources}")
print(f"Rows with missing source: {n_null_source}")
if not unique_sources <= {'yfinance', 'bloomberg'} or n_null_source > 0:
    issues_found.append(f"Source column integrity issue: values={unique_sources}, nulls={n_null_source}")
else:
    print("[PASS]")

# ------------------------------------------------------------
# C. Close vs adj_close_dividend_yfinance_unused DISTINCTNESS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("C. Close vs adj_close_dividend_yfinance_unused CHECK (yfinance rows)")
print("=" * 70)
yf_rows = df[df['source'] == 'yfinance']
both_present = yf_rows.dropna(subset=['Close', 'adj_close_dividend_yfinance_unused'])
pct_identical = (both_present['Close'] == both_present['adj_close_dividend_yfinance_unused']).mean() * 100
print(f"Of {len(both_present)} yfinance rows with both values present, "
      f"{pct_identical:.1f}% have IDENTICAL Close and adj_close values.")
print("(Expect well BELOW 100% -- near-100% would suggest an accidental duplication.)")
if pct_identical > 95:
    issues_found.append(f"Close and adj_close are suspiciously identical ({pct_identical:.1f}%) — investigate")
else:
    print("[PASS]")

bb_rows = df[df['source'] == 'bloomberg']
non_null_adjclose_bb = bb_rows['adj_close_dividend_yfinance_unused'].notna().sum()
print(f"\nBloomberg rows with a non-null adj_close (should be 0): {non_null_adjclose_bb}")
if non_null_adjclose_bb > 0:
    issues_found.append(f"Bloomberg rows unexpectedly have adj_close values: {non_null_adjclose_bb} rows")
else:
    print("[PASS]")

# ------------------------------------------------------------
# D. PIT FLAG ABSENCE (expected at this stage, not a bug)
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("D. PIT FLAG COLUMN CHECK (should be ABSENT at this stage)")
print("=" * 70)
pit_cols = [c for c in df.columns if 'nifty' in c.lower() or 'pit' in c.lower() or 'constituent' in c.lower()]
print(f"PIT-related columns found: {pit_cols} (expect: [])")
print("This is EXPECTED -- the PIT flag is joined separately at analysis")
print("time in the next stage, not baked into the raw price panel.")

# ------------------------------------------------------------
# E. TICKER COUNT RECONCILIATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("E. TICKER COUNT RECONCILIATION")
print("=" * 70)
n_panel_tickers = df['ticker'].nunique()
n_checkpoint_success = (latest_status == 'success').sum()
n_checkpoint_failed = (latest_status == 'failed').sum()
n_universe = len(pull_list)
print(f"Panel unique tickers:       {n_panel_tickers}")
print(f"Checkpoint 'success' count: {n_checkpoint_success}")
print(f"Checkpoint 'failed' count:  {n_checkpoint_failed}")
print(f"Total universe:             {n_universe}")
print(f"Success + failed = {n_checkpoint_success + n_checkpoint_failed} (should equal universe total)")
if n_panel_tickers != n_checkpoint_success:
    issues_found.append(f"Panel has {n_panel_tickers} tickers but checkpoint says {n_checkpoint_success} succeeded")
if n_checkpoint_success + n_checkpoint_failed != n_universe:
    issues_found.append("Checkpoint success+failed does not equal universe total")
if not issues_found or all('Panel' not in i and 'Checkpoint' not in i for i in issues_found):
    print("[PASS]")

# ------------------------------------------------------------
# F. INDEPENDENT RE-CHECK: duplicates, non-positive close, negative volume
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("F. INDEPENDENT DATA QUALITY RE-CHECK")
print("=" * 70)
dupes = df.duplicated(subset=['ticker', 'Date']).sum()
n_nonpositive = (df['Close'] <= 0).sum()
n_negvol = (df['Volume'] < 0).sum()
print(f"Duplicate (ticker, Date) rows: {dupes}")
print(f"Non-positive Close values: {n_nonpositive}")
print(f"Negative Volume values: {n_negvol}")
if dupes > 0 or n_nonpositive > 0 or n_negvol > 0:
    issues_found.append(f"Data quality issue: dupes={dupes}, nonpositive_close={n_nonpositive}, negvol={n_negvol}")
else:
    print("[PASS]")

# ------------------------------------------------------------
# G. SPARSE-TICKER RE-CHECK (confirm none remain below threshold)
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("G. SPARSE-TICKER RE-CHECK (threshold: 120 valid Close values)")
print("=" * 70)
MIN_VALID_ROWS = 120
valid_counts = df.groupby('ticker')['Close'].apply(lambda s: s.notna().sum())
still_sparse = valid_counts[valid_counts < MIN_VALID_ROWS]
print(f"Tickers still below the {MIN_VALID_ROWS}-day minimum: {len(still_sparse)}")
if not still_sparse.empty:
    print(still_sparse.to_string())
    issues_found.append(f"{len(still_sparse)} ticker(s) still below sparse threshold — sparse reclassification may not have run")
else:
    print("[PASS] — sparse-ticker reclassification fully took effect.")

# ------------------------------------------------------------
# H. NaN-RATE SUMMARY (informational, accepted, not blocking)
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("H. NaN-RATE SUMMARY (informational only -- these are ACCEPTED)")
print("=" * 70)
nan_rates = df.groupby('ticker')['Close'].apply(lambda s: s.isna().mean() * 100)
high_nan = nan_rates[nan_rates > 5].sort_values(ascending=False)
print(f"Tickers with >5% NaN Close (informational, already investigated this session): {len(high_nan)}")
if not high_nan.empty:
    print(high_nan.round(1).to_string())
print("\n(These reflect genuine illiquid/no-quote days for financially distressed")
print("names, confirmed via absolute valid-row-count analysis earlier this session")
print("-- not a pipeline bug. Documented in build_log.md.)")

# ------------------------------------------------------------
# I. PIT CROSS-VALIDATION RE-CHECK
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("I. PIT CROSS-VALIDATION RE-CHECK")
print("=" * 70)
canonical = pd.read_parquet(BASE + 'universe/canonical_membership_semiannual.parquet')
is_missing_row = canonical['exch_symbol'].isna() | (canonical['exch_symbol'] == '#N/A')
valid_membership = canonical[~is_missing_row]
isin_summary = (
    valid_membership.sort_values('snap_date').groupby('isin')
    .agg(exch_symbol=('exch_symbol', 'last'), true_first_appearance=('snap_date', 'min'))
    .reset_index()
)
isin_summary['ticker'] = isin_summary['exch_symbol'].str.strip() + '.NS'

price_starts = df.groupby('ticker')['Date'].min().reset_index()
price_starts.columns = ['ticker', 'actual_price_start']
merged = price_starts.merge(isin_summary[['ticker', 'true_first_appearance']], on='ticker', how='left')
merged['gap_days'] = (merged['actual_price_start'] - merged['true_first_appearance']).dt.days
still_flagged = merged[merged['gap_days'] > 180]
print(f"Tickers with a silent-gap flag remaining: {len(still_flagged)}")
if not still_flagged.empty:
    print(still_flagged.to_string(index=False))
    issues_found.append(f"{len(still_flagged)} ticker(s) still fail PIT cross-validation")
else:
    print("[PASS] — no silent gaps remain in the panel.")

# ------------------------------------------------------------
# J. NAMED SPOT-CHECK: the two known permanent gaps + the HINDMOTORS
#    recovery, confirmed directly against the data (not just the log).
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("J. NAMED SPOT-CHECK ON KNOWN PROBLEM TICKERS")
print("=" * 70)

for ticker, expected_status in [('MIRCELECTR.NS', 'failed'), ('DENABANK.NS', 'failed'), ('HINDMOTORS.NS', 'success')]:
    in_panel = ticker in df['ticker'].values
    actual_status = latest_status.get(ticker, 'NOT IN CHECKPOINT')
    print(f"\n{ticker}: checkpoint status = '{actual_status}' (expect '{expected_status}'), "
          f"present in consolidated panel = {in_panel}")
    if actual_status != expected_status:
        issues_found.append(f"{ticker}: expected status '{expected_status}', got '{actual_status}'")
    if expected_status == 'success' and not in_panel:
        issues_found.append(f"{ticker}: expected in panel (success) but not found")
    if expected_status == 'failed' and in_panel:
        issues_found.append(f"{ticker}: expected EXCLUDED (failed) but found in panel")
    if in_panel:
        sub = df[df['ticker'] == ticker]
        print(f"  Rows: {len(sub)}, valid Close: {sub['Close'].notna().sum()}, "
              f"date range: {sub['Date'].min()} to {sub['Date'].max()}, source: {sub['source'].iloc[0]}")

# ------------------------------------------------------------
# FINAL BANNER
# ------------------------------------------------------------
print("\n" + "=" * 70)
if issues_found:
    print(f"RESULT: {len(issues_found)} ISSUE(S) FOUND -- DO NOT treat Stage 3 as fully confirmed")
    print("=" * 70)
    for i in issues_found:
        print(f"  - {i}")
else:
    print("RESULT: ALL CHECKS PASSED -- Stage 3 is fully confirmed, ready for next stage")
print("=" * 70)

Checking: /content/drive/MyDrive/moonshot_compounding/raw/prices/raw_price_panel_consolidated_v1_2026-07-15.parquet

A. SCHEMA CONFORMANCE
Missing expected columns: set()
Unexpected extra columns: set()
[PASS]

B. SOURCE COLUMN INTEGRITY
Unique source values: {'bloomberg', 'yfinance'}
Rows with missing source: 0
[PASS]

C. Close vs adj_close_dividend_yfinance_unused CHECK (yfinance rows)
Of 3076620 yfinance rows with both values present, 22.3% have IDENTICAL Close and adj_close values.
(Expect well BELOW 100% -- near-100% would suggest an accidental duplication.)
[PASS]

Bloomberg rows with a non-null adj_close (should be 0): 0
[PASS]

D. PIT FLAG COLUMN CHECK (should be ABSENT at this stage)
PIT-related columns found: [] (expect: [])
This is EXPECTED -- the PIT flag is joined separately at analysis
time in the next stage, not baked into the raw price panel.

E. TICKER COUNT RECONCILIATION
Panel unique tickers:       1069
Checkpoint 'success' count: 1069
Checkpoint 'failed' count:  2
T

In [ ]:
# ======================================================================
# DIAGNOSE NaN PATTERN — for the 21 tickers QA flagged with >5% NaN Close.
#
# Do NOT assume these are all trailing-NaN-after-delisting. Check the
# actual position of the NaNs per ticker: trailing (delisting-like, safe
# to truncate same as we did for Bloomberg), leading (pre-listing-like,
# also generally safe), or scattered (a genuine data problem needing
# manual review, NOT safe to blanket-truncate).
# ======================================================================

import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

flagged_tickers = [
    'ASIANELEC.NS', 'BHARATFIN.NS', 'DENABANK.NS', 'DRDATSONS.NS',
    'EQUITAS.NS', 'ERAINFRA.NS', 'GAMMONIND.NS', 'ISEC.NS',
    'JMTAUTOLTD.NS', 'JPASSOCIAT.NS', 'JSLHISAR.NS', 'MAXINDIA.NS',
    'ROLTA.NS', 'SFCL.NS', 'SHILPI.NS', 'SINTEX.NS', 'STERLINBIO.NS',
    'TCNSBRANDS.NS', 'TECHNO.NS', 'UJJIVAN.NS', 'INNOIND.NS',
]

def classify_nan_pattern(close_series: pd.Series) -> str:
    is_nan = close_series.isna().values
    if not is_nan.any():
        return 'none'
    n = len(is_nan)
    first_valid = np.argmax(~is_nan)
    last_valid = n - 1 - np.argmax(~is_nan[::-1])
    edge_mask = np.zeros(n, dtype=bool)
    edge_mask[:first_valid] = True
    edge_mask[last_valid + 1:] = True
    nan_positions = np.where(is_nan)[0]
    all_edge = np.all(edge_mask[nan_positions])
    if not all_edge:
        return 'scattered'
    has_leading = is_nan[:first_valid].sum() > 0
    has_trailing = is_nan[last_valid + 1:].sum() > 0
    if has_leading and has_trailing:
        return 'both_edges'
    elif has_trailing:
        return 'trailing'
    elif has_leading:
        return 'leading'
    return 'none'

results = []
for ticker in flagged_tickers:
    fpath = PRICE_BATCH_DIR + ticker.replace('.', '_') + '.parquet'
    df = pd.read_parquet(fpath).sort_values('Date').reset_index(drop=True)
    pattern = classify_nan_pattern(df['Close'])

    valid = df.dropna(subset=['Close'])
    last_valid_date = valid['Date'].max() if not valid.empty else None
    date_max = df['Date'].max()

    results.append({
        'ticker': ticker,
        'pattern': pattern,
        'pct_nan': df['Close'].isna().mean() * 100,
        'last_valid_date': last_valid_date,
        'file_date_max': date_max,
        'trailing_gap_days': (date_max - last_valid_date).days if last_valid_date is not None else None,
    })

results_df = pd.DataFrame(results)
print("=" * 70)
print("NaN PATTERN CLASSIFICATION FOR 21 FLAGGED TICKERS")
print("=" * 70)
print(results_df.to_string(index=False))

print(f"\nPattern breakdown:")
print(results_df['pattern'].value_counts())

scattered = results_df[results_df['pattern'] == 'scattered']
if not scattered.empty:
    print(f"\n[FLAG FOR MANUAL REVIEW — do NOT auto-truncate] {len(scattered)} ticker(s) "
          f"have SCATTERED NaN (not just at the edges) — this is a different, more "
          f"concerning problem than delisting:")
    print(scattered[['ticker', 'pct_nan']].to_string(index=False))

results_df.to_csv(BASE + 'validation/nan_pattern_diagnosis.csv', index=False)
print(f"\nSaved to validation/nan_pattern_diagnosis.csv")

NaN PATTERN CLASSIFICATION FOR 21 FLAGGED TICKERS
       ticker   pattern   pct_nan last_valid_date file_date_max  trailing_gap_days
 ASIANELEC.NS scattered 20.710660      2013-12-09    2013-12-09                  0
 BHARATFIN.NS   leading  6.533729      2019-07-02    2019-07-02                  0
  DENABANK.NS scattered 97.345133      2010-11-24    2010-11-24                  0
 DRDATSONS.NS   leading 26.138910      2015-05-21    2015-05-21                  0
   EQUITAS.NS   leading 48.107110      2023-02-02    2023-02-02                  0
  ERAINFRA.NS scattered  6.430288      2016-09-12    2016-09-12                  0
 GAMMONIND.NS scattered  5.046296      2018-09-10    2018-09-10                  0
      ISEC.NS   leading 54.208576      2025-03-21    2025-03-21                  0
JMTAUTOLTD.NS scattered  6.229826      2022-06-27    2022-06-27                  0
JPASSOCIAT.NS scattered  5.643958      2026-03-17    2026-03-17                  0
  JSLHISAR.NS   leading 46.149144    

In [ ]:
# ======================================================================
# ABSOLUTE VALID-ROW COUNT for the 21 NaN-flagged tickers.
#
# WHY: percentage NaN alone is misleading. 97% NaN on a 226-row file
# leaves ~6 usable days (unusable for any feature). 5% NaN on a 4000-row
# file leaves ~3800 usable days (fine). We need the ABSOLUTE count to
# decide which of these 21 are genuinely usable-with-caveats vs which
# should be reclassified as failed, same treatment as the earlier
# phantom-quote cases.
#
# Threshold: need at least 120 valid trading days to support the longest
# planned feature (realized_vol_120). Anything below that can't support
# the full feature set regardless of what else is fixed.
# ======================================================================

import pandas as pd

BASE = '/content/drive/MyDrive/moonshot_compounding/'
PRICE_BATCH_DIR = BASE + 'raw/prices/batches/'

flagged_tickers = [
    'ASIANELEC.NS', 'BHARATFIN.NS', 'DENABANK.NS', 'DRDATSONS.NS',
    'EQUITAS.NS', 'ERAINFRA.NS', 'GAMMONIND.NS', 'ISEC.NS',
    'JMTAUTOLTD.NS', 'JPASSOCIAT.NS', 'JSLHISAR.NS', 'MAXINDIA.NS',
    'ROLTA.NS', 'SFCL.NS', 'SHILPI.NS', 'SINTEX.NS', 'STERLINBIO.NS',
    'TCNSBRANDS.NS', 'TECHNO.NS', 'UJJIVAN.NS', 'INNOIND.NS',
]

MIN_VALID_ROWS = 120

results = []
for ticker in flagged_tickers:
    fpath = PRICE_BATCH_DIR + ticker.replace('.', '_') + '.parquet'
    df = pd.read_parquet(fpath)
    n_total = len(df)
    n_valid = df['Close'].notna().sum()
    results.append({
        'ticker': ticker,
        'source': df['source'].iloc[0] if 'source' in df.columns else 'yfinance',
        'n_total_rows': n_total,
        'n_valid_close': n_valid,
        'pct_nan': (1 - n_valid / n_total) * 100 if n_total > 0 else 100,
        'below_min_threshold': n_valid < MIN_VALID_ROWS,
    })

results_df = pd.DataFrame(results).sort_values('n_valid_close')
print("=" * 70)
print(f"ABSOLUTE VALID-ROW COUNTS (threshold: {MIN_VALID_ROWS} valid days minimum)")
print("=" * 70)
print(results_df.to_string(index=False))

below = results_df[results_df['below_min_threshold']]
print(f"\n{len(below)} ticker(s) fall BELOW the {MIN_VALID_ROWS}-day minimum "
      f"(candidates for reclassification to failed):")
if not below.empty:
    print(below[['ticker', 'n_valid_close']].to_string(index=False))

above = results_df[~results_df['below_min_threshold']]
print(f"\n{len(above)} ticker(s) have enough valid data to keep, WITH the NaN")
print(f"days documented (they'll correctly propagate to NaN in derived features")
print(f"on those specific dates -- honest behavior, not a bug):")
if not above.empty:
    print(above[['ticker', 'n_valid_close', 'pct_nan']].to_string(index=False))

results_df.to_csv(BASE + 'validation/nan_usability_assessment.csv', index=False)
print(f"\nSaved to validation/nan_usability_assessment.csv")

ABSOLUTE VALID-ROW COUNTS (threshold: 120 valid days minimum)
       ticker    source  n_total_rows  n_valid_close   pct_nan  below_min_threshold
  DENABANK.NS bloomberg           226              6 97.345133                 True
 ASIANELEC.NS bloomberg           985            781 20.710660                False
      SFCL.NS bloomberg          1800            844 53.111111                False
  MAXINDIA.NS bloomberg          2588            965 62.712519                False
 DRDATSONS.NS bloomberg          1339            989 26.138910                False
   INNOIND.NS bloomberg          1602           1144 28.589263                False
TCNSBRANDS.NS bloomberg          3639           1509 58.532564                False
  ERAINFRA.NS bloomberg          1664           1557  6.430288                False
   EQUITAS.NS bloomberg          3249           1686 48.107110                False
      ISEC.NS bloomberg          3778           1730 54.208576                False
  JSLHISAR.NS 